In [1]:
import os
from dotenv import load_dotenv

from openai import OpenAI
import sqlite3
import gradio as gr
import json

c:\Users\LENOVO\Documents\projects\llm_mini_projects\personal-expense-tracker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if GOOGLE_API_KEY is not None:
    print("API Key Fetched")

API Key Fetched


In [14]:
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=GOOGLE_API_KEY)

In [42]:
# Open a db connection
DB_NAME = "expense.db"

with sqlite3.connect(DB_NAME) as conn:
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS expense (
            id INTEGER PRIMARY KEY AUTOINCREMENT, 
            description TEXT NOT NULL, 
            amount REAL NOT NULL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()

In [53]:
# tool to add expense
def add_expense(description: str, amount: float) -> None:
    print("Calling add expense tool !")
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO expense (description, amount) VALUES
            (?, ?)
        """, (description, amount, ))
        try:
            conn.commit()
            print(f"{description}:{amount} Added successfully !")
        except:
            print(f"Could not add {description}:{amount}")

In [54]:
# tool to retrieve all existing expenses
def retrieve_all_expenses() -> list:
    print("Calling retrieve all expenses tool !")
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        try:
            expenses = cursor.fetchall()
            print("Fetched expenses successfully !")
        except:
            print("Could not retrieve expenses")
    return json.dumps(expenses)

In [55]:
add_expense_to_db = {
    "name": "add_expense",
    "description": "Add a single expense to the database",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "A one line short description about the expense made that is to be added to the database"
            },
            "amount": {
                "type": "number",
                "description": "The amount spent that is to be added to the database"
            }
        },
        "required": ["description", "amount"],
    }
}

In [56]:
retrieve_all_expenses_from_db = {
    "name": "retrieve_all_expenses",
    "description": "Retrieve all expenses added to the database",
    "parameters": {
        "type": "object",
        "properties": {}
    }
}

In [57]:
tools = [
    {"type": "function", "function": add_expense_to_db},
    {"type": "function", "function": retrieve_all_expenses_from_db}
]

In [58]:
SYSTEM_PROMPT = """
You are an assistant who manages the expenses made by a user. You can store them in a database, 
retrieve them from a database (you will have tools to do so). You should also analyze the 
expenses from their descriptions and amounts and chart out suggestions to be made to the user
as and when asked for. 

You WILL NOT hallucinate any expense that is not present in the database and return null if the 
expense does not exist.

When displaying the expenses, you shall do so in a structured manner and respond to the user with
appropriate messages
"""

In [59]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(
        model="gemini-3.5-flash-lite",
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_response = handle_tool_call(message)
        messages.append(message)
        messages.append(tool_response)
        response = gemini.chat.completions.create(
            model="gemini-3.5-flash-lite",
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

In [60]:
def handle_tool_call(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "add_expense":
            arguments = json.loads(tool_call.function.arguments)
            description = arguments.get("description")
            amount = arguments.get("description")
            try:
                add_expense(description, amount)
                response = {
                    "role": "tool",
                    "content": f"Successfully added expense of {amount} for {description}",
                    "tool_call_id": tool_call.id
                }
            except:
                response = {
                    "role": "tool",
                    "content": f"ERROR: Could not add expense of {amount} for {description}",
                    "tool_call_id": tool_call.id
                }
            responses.append(response)
        elif tool_call.function.name == "retrieve_all_expenses_from_db":
            try:
                all_expenses = retrieve_all_expenses()
                response = {
                    "role": "tool",
                    "content": all_expenses,
                    "tool_call_id": tool_call.id
                }
            except:
                response = {
                    "role": "tool",
                    "content": "ERROR: Could not fetch expenses",
                    "tool_call_id": tool_call.id
                }
            responses.append(response)
        else:
            responses.append({
                "role": "tool",
                "content": "No such tool found",
                "tool_call_id": tool_call.id
            })

    return responses

In [ ]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


Calling add expense tool !
Starbucks:Starbucks Added successfully !
Calling add expense tool !
Clothes:Clothes Added successfully !
Calling add expense tool !
Food:Food Added successfully !
Calling add expense tool !
Chaats:Chaats Added successfully !
